# 19 · The told model: how exact the QP has to be, and how it ships

Notebook 18 trained notebook 7's body told each patch's slice QP. Two questions before it can ship:

- **A wrong QP.** The model given the QP off by a few steps: how exact the number has to be. The client reads it off
  the slice header, so it is exact; a client that baked a few graphs and picked the nearest would be off by a few.
- **Biases that are a line in the QP.** Each shift is a 1×1 convolution of the QP with no bias, so a graph baked for
  one QP has the biases `b + w * QP / 51`. The client can compute them every frame from the QP it reads, with the
  blind graph's cost, instead of picking from a few baked graphs.

This notebook trains nothing: it loads notebook 18's told runs and scores them on notebook 0's validation split.

Needs: `18_upscale_qp_models.ipynb` (its runs in `checkpoints/qp_runs/`).

In [ ]:
"""Settings."""

import os

import numpy as np
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, training
from helpers.training import DATA_DIR, DEVICE

WIDTH, BLOCKS, STEM = 24, 2, 3               # notebook 7's body
QP_MAX = patches.QP_MAX                      # the model is told QP / QP_MAX
QP_ERRORS = (-12, -6, -3, 3, 6, 12)          # how far off the told model is told, in QP steps
RUNS = {seed: training.CHECKPOINT_DIR / "qp_runs" / f"told_qp_seed{seed}.pt" for seed in (0, 1)}   # notebook 18's
COLUMNS = patches.columns(("qp",))

manifest = training.load_manifest()
SCALE = manifest["scale"]


def told_model():
    return models.ToldLuma(1, WIDTH, BLOCKS, STEM, scale=SCALE).to(DEVICE)


def biases(model, qp):
    """The three biases a graph baked from told `model` for `qp` has, end to end."""
    plain = model.baked(qp / QP_MAX)
    return torch.cat([plain.body[0].bias, *[block.conv.bias for block in plain.body[2:]]])

## 1. The biases are a line in the QP

`ToldLuma.baked(c)` adds `shift(c)` to three biases, and each shift is a 1×1 convolution of one number with no bias of
its own: `w * c`. So a graph baked for a QP has the biases `b + w * QP / 51`, with `w` 24 numbers for each of the
three. Checked on random weights: a graph baked at any QP has the biases of the line through QP 0 and QP 51, and
computes what the told model computes.

In [ ]:
probe = told_model()
with torch.no_grad(), training.full_precision():
    for weight in probe.parameters():
        if weight.requires_grad:
            weight.normal_(0, 0.1)
    at_0, at_max = biases(probe, 0), biases(probe, QP_MAX)
    x = torch.rand(2, 3, 40, 40, device=DEVICE)
    for qp in (7, 23.5, 41):
        off_line = (biases(probe, qp) - (at_0 + (at_max - at_0) * qp / QP_MAX)).abs().max().item()
        computes = (probe(x, torch.full((2, 1, 1, 1), qp / QP_MAX, device=DEVICE))
                    - probe.baked(qp / QP_MAX)(x)).abs().max().item()
        assert off_line < 1e-5, f"baked at QP {qp}, the biases are off the line by {off_line:.1e}"
        assert computes < 1e-5, f"baked at QP {qp}, the graph computes something else ({computes:.1e})"
print(f"a graph baked for a QP: biases b + w * QP / {QP_MAX}, {at_0.numel()} of them, and the told model's output")

## 2. A wrong QP

The told model at the seed that did better on the select split, given every patch's QP off by each of `QP_ERRORS`
(kept inside 0 to 51): SSIM-Y and luma dB against the same model told the true QP, by slice QP bin. Every +6 doubles
the quantiser's step.

In [ ]:
select_scores = {seed: torch.load(path, map_location="cpu", weights_only=True)["best"]["score"]
                 for seed, path in RUNS.items()}
seed = max(select_scores, key=select_scores.get)
model = told_model()
model.load_state_dict(training.fitted_weights(RUNS[seed]))
model.eval()


def told_off_by(error):
    return lambda lr, condition: model(lr, (condition[:, COLUMNS] + error / QP_MAX).clamp(0, 1))


true = training.val_scores(told_off_by(0), manifest)
wrong = {error: training.val_scores(told_off_by(error), manifest) for error in QP_ERRORS}

qp = np.load(DATA_DIR / manifest["splits"]["val"]["qp"])
edges = manifest["qp_edges"]
print(f"told the QP, seed {seed}; error in QP steps: SSIM-Y / luma dB against the true QP")
print(f"{'QP':>7}{'patches':>9}" + "".join(f"{error:>+16d}" for error in QP_ERRORS))
for low, high in [(0, 99), *zip(edges[:-1], edges[1:])]:
    inside = (qp >= low) & (qp < high)
    cells = [f"{(w['ssim'] - true['ssim'])[inside].mean():+.4f} / {(w['y'] - true['y'])[inside].mean():+.2f}"
             for w in wrong.values()]
    print(f"{'all' if high == 99 else f'{low}-{high}':>7}{inside.sum():9}" + "".join(f"{cell:>16}" for cell in cells))

## 3. How much the model leans on the QP

How far the QP moves each layer's biases from QP 0 to 51, against their size at QP 0.

In [ ]:
with torch.no_grad():
    at_0, at_max = biases(model, 0), biases(model, QP_MAX)
for layer, start in zip(("stem", "block 1", "block 2"), (0, WIDTH, 2 * WIDTH)):
    moved, size = (at_max - at_0)[start:start + WIDTH].abs(), at_0[start:start + WIDTH].abs()
    print(f"{layer:<8} QP 0 to {QP_MAX} moves a bias {moved.mean():.3f} on average (largest {moved.max():.3f}); "
          f"the biases at QP 0 average {size.mean():.3f}")

## 4. What it says

*(written after the run)*